# Wrong-Answer Tier Agreement (Pillar 2, PathOPEN only)

Implements the paper's Pillar 2 methodology (Section 2.2 / Table 2): PathOPEN's
pathologist-authored wrong answers form a graded difficulty structure
(near-miss / moderate-miss / far-miss), which this notebook validates via
judge-agreement analysis, entirely within PathOPEN (no comparator dataset - PathVQA
has no wrong answers at all).

**Tiering** (per the paper): existing pathologist Benchmark 2 scores (Error
Proximity and Deductive Plausibility; Visual Grounding Error) are binned as:
- score = 2 -> **near-miss**
- score = 1 -> **moderate-miss**
- score = 0 -> **far-miss**
- score = -1 -> excluded ("unable to comprehend / evaluate")

This is done separately for **OE-native** wrong answers (`OE_Wrong_Answer_{1,2}`)
and **MCQ-derived** wrong answers (`MCQ_OE_Wrong_Answer_{1-4}`), per the paper's
explicit distinction (Figure 4 Panel A: "Tier distribution (OE-native vs.
MCQ-derived)").

The validated judge model(s) re-score the same wrong-answer items on Benchmark 2,
tiered the same way, and this notebook reports:
- **Tier-assignment agreement**: weighted kappa + 3x3 confusion matrix
  (judge tier vs. human tier), per criterion (Error Proximity, Visual Grounding
  Error) and per source (OE-native, MCQ-derived).
- **Per-tier sample sizes**, with any tier n < 30 flagged, per the paper's own
  stated reviewer concern.

**Prerequisite**: run `judge_runner_pathopen.ipynb` first so
`judge_output/evaluator_internvl/pathopen_eval_data.csv` and
`judge_output/evaluator_qwenvl/pathopen_eval_data.csv` exist.


In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix


In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
HUMAN_INPUT_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "scoring_analysis", "input"
)
JUDGE_OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
JUDGE_KEYS = ["internvl", "qwenvl"]

TIER_LABELS = {2: "near-miss", 1: "moderate-miss", 0: "far-miss"}
MIN_TIER_N = 30  # paper's own stated flag threshold

REPO_ROOT, HUMAN_INPUT_DIR, JUDGE_OUTPUT_DIR


In [ ]:
def _evaluator_dirs_in_order() -> list:
    dirs = glob.glob(os.path.join(HUMAN_INPUT_DIR, "evaluator[0-9]*"))
    return sorted(dirs, key=lambda p: int(os.path.basename(p).replace("evaluator", "")))


def load_pooled_human(dataset_filename: str) -> pd.DataFrame:
    frames = []
    for evaluator_dir in _evaluator_dirs_in_order():
        path = os.path.join(evaluator_dir, dataset_filename)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df = df.drop(index=0).reset_index(drop=True)  # drop benchmark-label header row
        df["__evaluator__"] = os.path.basename(evaluator_dir)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


human_pathopen = load_pooled_human("pathopen_eval_data.csv")

judge_pathopen = {
    key: pd.read_csv(os.path.join(JUDGE_OUTPUT_DIR, f"evaluator_{key}", "pathopen_eval_data.csv"))
    for key in JUDGE_KEYS
}

human_pathopen.shape, {k: v.shape for k, v in judge_pathopen.items()}


## Wrong-answer column map

Each entry: `(source, criterion, human_column, judge_column)`. `source` is
`OE_native` or `MCQ_derived`, matching the paper's Figure 4 Panel A distinction.

In [ ]:
WRONG_ANSWER_COLUMN_MAP = []
for i in (1, 2):
    WRONG_ANSWER_COLUMN_MAP.append((
        "OE_native", "Error Proximity and Deductive Plausibility",
        f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
        f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
    ))
    WRONG_ANSWER_COLUMN_MAP.append((
        "OE_native", "Visual Grounding Error",
        f"Unnamed: {9 if i == 1 else 16}",
        f"OE_Wrong_Answer_{i}_VisGroundErr",
    ))

for i in (1, 2, 3, 4):
    unnamed_idx = {1: 23, 2: 26, 3: 29, 4: 32}[i]
    WRONG_ANSWER_COLUMN_MAP.append((
        "MCQ_derived", "Error Proximity and Deductive Plausibility",
        f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
        f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
    ))
    WRONG_ANSWER_COLUMN_MAP.append((
        "MCQ_derived", "Visual Grounding Error",
        f"Unnamed: {unnamed_idx}",
        f"MCQ_OE_Wrong_Answer_{i}_VisGroundErr",
    ))

len(WRONG_ANSWER_COLUMN_MAP)


## Tier binning + agreement

`Image_ID` is unique across the pooled PathOPEN human dataset (verified separately
in `judge_pathologist_agreement.ipynb`), so it is a safe join key here too.

### Why `-1` is excluded here, unlike everywhere else

`judge_pathologist_agreement.ipynb` and `quiltvqa_eval_runner.ipynb` deliberately
**include** `-1` in their kappa and Mann-Whitney statistics, because it is a real
rubric level ("unable to comprehend / evaluate") carrying a dataset-quality signal.

This notebook is the one place that still drops it, and for a structural reason
rather than a statistical preference: Pillar 2 maps scores onto a **3-tier scale**
(2 -> near-miss, 1 -> moderate-miss, 0 -> far-miss), and `-1` has no tier. The paper
defines the binning that way. Keeping `-1` would require inventing a fourth tier the
paper does not define, and would make the 3x3 confusion matrix ill-formed.

So the exclusion here is part of the tier definition, not a filtering choice - but it
does mean the tier analysis is silent about unscorable wrong answers. The count of
`-1`s dropped is worth reporting alongside the tier distribution, since a wrong answer
the pathologist could not interpret is itself a finding about the item.

In [ ]:
VALID_SCORES = {-1, 0, 1, 2}


def _resolve_column(merged: pd.DataFrame, col: str, side: str) -> pd.Series:
    suffixed = f"{col}_{side}"
    if suffixed in merged.columns:
        return pd.to_numeric(merged[suffixed], errors="coerce")
    return pd.to_numeric(merged[col], errors="coerce")


def tier_agreement_for_column(human_df, judge_df, human_col, judge_col):
    merged = human_df.merge(judge_df, on="Image_ID", suffixes=("_human", "_judge"))
    human_scores = _resolve_column(merged, human_col, "human")
    judge_scores = _resolve_column(merged, judge_col, "judge")

    valid_mask = human_scores.isin({0, 1, 2}) & judge_scores.isin(VALID_SCORES)
    human_valid = human_scores[valid_mask].astype(int)
    judge_valid = judge_scores[valid_mask].astype(int)
    # Judge -1 ("unable to evaluate") has no tier; exclude those pairs too, keeping
    # the human's tier scale (0,1,2) as ground truth for the confusion matrix.
    judge_has_tier = judge_valid != -1
    human_valid = human_valid[judge_has_tier]
    judge_valid = judge_valid[judge_has_tier]

    n = len(human_valid)
    tier_counts = human_valid.value_counts().reindex([2, 1, 0], fill_value=0)
    flagged_tiers = [TIER_LABELS[t] for t, c in tier_counts.items() if c < MIN_TIER_N]

    if n < 2:
        kappa = np.nan
        cm = None
    else:
        kappa = cohen_kappa_score(human_valid, judge_valid, weights="quadratic")
        cm = confusion_matrix(human_valid, judge_valid, labels=[2, 1, 0])

    return {
        "n": n,
        "weighted_kappa": kappa,
        "confusion_matrix": cm,
        "tier_counts": tier_counts.to_dict(),
        "flagged_low_n_tiers": flagged_tiers,
    }


In [ ]:
tier_results = []
for model_key in JUDGE_KEYS:
    for source, criterion, human_col, judge_col in WRONG_ANSWER_COLUMN_MAP:
        if human_col not in human_pathopen.columns or judge_col not in judge_pathopen[model_key].columns:
            continue
        result = tier_agreement_for_column(human_pathopen, judge_pathopen[model_key], human_col, judge_col)
        tier_results.append({
            "judge": model_key,
            "source": source,
            "criterion": criterion,
            **{k: v for k, v in result.items() if k != "confusion_matrix"},
        })

tier_results_df = pd.DataFrame(tier_results)
tier_results_df


In [ ]:
os.makedirs(os.path.join(os.getcwd(), "agreement_output"), exist_ok=True)
tier_results_df.to_csv(os.path.join(os.getcwd(), "agreement_output", "wrong_answer_tier_agreement.csv"), index=False)


## Confusion matrices (per judge x source x criterion)

Printed separately since a 3x3 matrix doesn't fit cleanly in the summary table
above. Rows = human tier (near-miss, moderate-miss, far-miss), columns = judge
tier, in that same order.

In [ ]:
for model_key in JUDGE_KEYS:
    for source, criterion, human_col, judge_col in WRONG_ANSWER_COLUMN_MAP:
        if human_col not in human_pathopen.columns or judge_col not in judge_pathopen[model_key].columns:
            continue
        result = tier_agreement_for_column(human_pathopen, judge_pathopen[model_key], human_col, judge_col)
        if result["confusion_matrix"] is None:
            continue
        print(f"\n{model_key} | {source} | {criterion} (n={result['n']}, weighted kappa={result['weighted_kappa']:.3f})")
        cm_df = pd.DataFrame(
            result["confusion_matrix"],
            index=[f"human_{TIER_LABELS[t]}" for t in (2, 1, 0)],
            columns=[f"judge_{TIER_LABELS[t]}" for t in (2, 1, 0)],
        )
        print(cm_df)
        if result["flagged_low_n_tiers"]:
            print(f"  FLAGGED (n < {MIN_TIER_N}): {result['flagged_low_n_tiers']}")


## Optional: monotonic difficulty gradient (Figure 4 Panel C)

The paper notes this as an *optional* behavioral extension: test whether zero-shot
model accuracy at distinguishing correct-from-wrong decreases monotonically across
tiers (near-miss should be hardest to distinguish, far-miss easiest). This requires
running a candidate VLM zero-shot on a correct-vs-wrong discrimination task, which
is answerer-benchmarking (Pillar 3b/4 territory), not judge-scoring - out of scope
for this notebook. Flagging here as a natural follow-up once the answerer-benchmarking
phase begins.